In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.metrics         import (accuracy_score, f1_score, precision_score,
                                     recall_score, roc_auc_score,
                                     average_precision_score, confusion_matrix,
                                     precision_recall_curve, roc_curve)
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)
print("All libraries loaded!")

In [ ]:
data = pd.read_csv("smart_grid_stability_augmented.csv")

data = data.rename(columns={
    "tau1" : "reaction_producer",
    "tau2" : "reaction_consumer1",
    "tau3" : "reaction_consumer2",
    "tau4" : "reaction_consumer3",
    "p1"   : "power_producer",
    "p2"   : "power_consumer1",
    "p3"   : "power_consumer2",
    "p4"   : "power_consumer3",
    "g1"   : "price_sensitivity_producer",
    "g2"   : "price_sensitivity_consumer1",
    "g3"   : "price_sensitivity_consumer2",
    "g4"   : "price_sensitivity_consumer3",
    "stab" : "stability_score",
    "stabf": "grid_status",
})

print(f"Rows    : {len(data):,}")
print(f"Columns : {list(data.columns)}")
data.head()

In [ ]:
print("Missing values  :", data.isnull().sum().sum(),  "← should be 0")
print("Duplicate rows  :", data.duplicated().sum(),    "← should be 0")
print()
print("Label counts:")
print(data["grid_status"].value_counts())

In [ ]:
plt.figure(figsize=(16, 10))
plt.suptitle("EDA — Understanding the Grid Stability Data", fontsize=14)

# ── Chart 1: Stable vs Unstable count ───────────────────
plt.subplot(2, 3, 1)
counts = data["grid_status"].value_counts()
plt.bar(counts.index, counts.values, color=["steelblue", "tomato"])
plt.title("Stable vs Unstable Count")
plt.ylabel("Number of rows")
for i, val in enumerate(counts.values):
    plt.text(i, val + 200, f"{val:,}", ha="center", fontweight="bold")

In [ ]:
# ── Chart 2: Stability score by class ───────────────────
# Negative score = unstable, positive = stable
plt.subplot(2, 3, 2)
sns.histplot(data[data["grid_status"]=="stable"]["stability_score"],
             bins=50, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["stability_score"],
             bins=50, color="tomato", alpha=0.6, label="Unstable")
plt.axvline(0, color="black", linestyle="--", label="Boundary = 0")
plt.title("Stability Score by Class")
plt.xlabel("stability_score")
plt.legend()

# ── Chart 3: Reaction producer by class ─────────────────
plt.subplot(2, 3, 3)
sns.histplot(data[data["grid_status"]=="stable"]["reaction_producer"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["reaction_producer"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Reaction Producer by Class")
plt.xlabel("reaction_producer")
plt.legend()


In [ ]:
# ── Chart 4: Power producer by class ────────────────────
plt.subplot(2, 3, 4)
sns.histplot(data[data["grid_status"]=="stable"]["power_producer"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["power_producer"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Power Producer by Class")
plt.xlabel("power_producer")
plt.legend()

# ── Chart 5: Reaction consumer1 by class ────────────────
plt.subplot(2, 3, 5)
sns.histplot(data[data["grid_status"]=="stable"]["reaction_consumer1"],
             bins=40, color="steelblue", alpha=0.6, label="Stable")
sns.histplot(data[data["grid_status"]=="unstable"]["reaction_consumer1"],
             bins=40, color="tomato", alpha=0.6, label="Unstable")
plt.title("Reaction Consumer1 by Class")
plt.xlabel("reaction_consumer1")
plt.legend()

In [ ]:
# ── Chart 6: Correlation heatmap ─────────────────────────
plt.subplot(2, 3, 6)
cols_for_corr = ["reaction_producer", "power_producer",
                 "price_sensitivity_producer", "stability_score"]
corr = data[cols_for_corr].corr().round(2)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5)
plt.title("Correlation Heatmap")

plt.tight_layout()
plt.savefig("outputs/01_eda.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: outputs/01_eda.png")

In [ ]:
input_features = [
    "reaction_producer",  "reaction_consumer1", "reaction_consumer2", "reaction_consumer3",
    "power_producer",     "power_consumer1",    "power_consumer2",    "power_consumer3",
    "price_sensitivity_producer",  "price_sensitivity_consumer1",
    "price_sensitivity_consumer2", "price_sensitivity_consumer3",
]

X = data[input_features]
y = (data["grid_status"] == "unstable").astype(int)   # 1 = unstable, 0 = stable

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Training rows : {len(X_train):,}")
print(f"Testing rows  : {len(X_test):,}")

In [ ]:
# Create the 5 models
log_reg  = LogisticRegression(max_iter=1000, random_state=42)
dec_tree = DecisionTreeClassifier(max_depth=8, random_state=42)
ran_for  = RandomForestClassifier(n_estimators=100, random_state=42)
grad_bst = GradientBoostingClassifier(n_estimators=100, random_state=42)
knn      = KNeighborsClassifier(n_neighbors=7)

# Train each model
# Logistic Regression and KNN need scaled data
# Tree models work on original data
log_reg.fit(X_train_scaled, y_train)
dec_tree.fit(X_train, y_train)
ran_for.fit(X_train, y_train)
grad_bst.fit(X_train, y_train)
knn.fit(X_train_scaled, y_train)

print("All 5 models trained!")

In [ ]:
# Get predictions (0 or 1) and probabilities (0.0 to 1.0) for each model
pred_log  = log_reg.predict(X_test_scaled)
pred_dec  = dec_tree.predict(X_test)
pred_ran  = ran_for.predict(X_test)
pred_grad = grad_bst.predict(X_test)
pred_knn  = knn.predict(X_test_scaled)

prob_log  = log_reg.predict_proba(X_test_scaled)[:, 1]
prob_dec  = dec_tree.predict_proba(X_test)[:, 1]
prob_ran  = ran_for.predict_proba(X_test)[:, 1]
prob_grad = grad_bst.predict_proba(X_test)[:, 1]
prob_knn  = knn.predict_proba(X_test_scaled)[:, 1]

# Score each model
model_names = ["Logistic Regression", "Decision Tree",
               "Random Forest", "Gradient Boosting", "K-Nearest Neighbors"]
all_preds   = [pred_log,  pred_dec,  pred_ran,  pred_grad,  pred_knn]
all_probas  = [prob_log,  prob_dec,  prob_ran,  prob_grad,  prob_knn]

acc_scores  = [accuracy_score(y_test, p)          for p in all_preds]
f1_scores   = [f1_score(y_test, p)                for p in all_preds]
prec_scores = [precision_score(y_test, p)         for p in all_preds]
rec_scores  = [recall_score(y_test, p)            for p in all_preds]
roc_scores  = [roc_auc_score(y_test, p)           for p in all_probas]
ap_scores   = [average_precision_score(y_test, p) for p in all_probas]

print(f"{'Model':<25} {'Accuracy':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'ROC-AUC':>9}")
print("-" * 72)
for i, name in enumerate(model_names):
    print(f"{name:<25} {acc_scores[i]:>10.2%} {f1_scores[i]:>8.4f}"
          f" {prec_scores[i]:>10.4f} {rec_scores[i]:>8.4f} {roc_scores[i]:>9.4f}")

best_idx  = f1_scores.index(max(f1_scores))
best_name = model_names[best_idx]
print(f"\nBest model: {best_name}  (F1 = {max(f1_scores):.4f})")